In [29]:
#!pip install crewai crewai_tools==0.1.6 langchain_community==0.0.29
!pip install poetry

!pip install crewai 
!pip install crewai crewai-tools

In [30]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [31]:
from crewai import Agent, Task,Crew

import os
from langchain.llms import Ollama
'''from utils import get_openai_api_key
openai_api_key= get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"]='gpt-4o'
'''
lama_llm = Ollama(
    model = "llama2",
    base_url = "http://localhost:11434")


In [32]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic."
              "You also collect sample how to examples on {topic} that Content Writer can use for the content.",
    allow_delegation=False,
    llm=lama_llm,
	verbose=True
)

In [33]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=lama_llm,
    verbose=True
)

In [34]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=True,
    llm=lama_llm,
    verbose=True
)
editor

Agent(role=Editor, goal=Edit a given blog post to align with the writing style of the organization. , backstory=You are an editor who receives a blog post from the Content Writer. Your goal is to review the blog post to ensure that it follows journalistic best practices,provides balanced viewpoints when providing opinions or assertions, and also avoids major controversial topics or opinions when possible.)

In [35]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

In [36]:
write= Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
        "6. include all the refrences as link at the end of article.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have at least 2 or 3 paragraphs and each paragraph being at least 200 words."
        "Include links to the reference at the end  of sentence or paragraph."
        "include the full list of references at the end of the article",
    agent=writer,
)

In [37]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."
                 "include all the refrences as link at the end of article"),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have at least 2 or 3 paragraphs and each paragraph being at least 200 words."
                    "the full list of references must be presented at the end of the article",
    agent=editor,
)

In [38]:
crew= Crew(agents=[planner,writer,editor],
           tasks=[plan,write,edit],
           verbose=2
        )

In [44]:

result = crew.kickoff(inputs={"topic": "AWS EC2"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on AWS EC2.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
Final Answer:

As a Content Planner, I have conducted thorough research on the latest trends, key players, and noteworthy news related to AWS EC2. Based on my analysis, I prioritized the following information to be included in the content plan document:

1. Introduction: A brief overview of AWS EC2 and its significance in the cloud computing industry.
2. Key Players: An analysis of the major players in the AWS EC2 market, including Amazon Web Services (AWS), Microsoft Azure, Google Cloud Platform (GCP), and IBM Cloud. This section will highlight their strengths, weaknes

2024-10-14 03:18:11,047 - 6425997312 - __init__.py-__init__:369 - ERROR: Exception while exporting Span batch.
Traceback (most recent call last):
  File "/Users/LearningUser/Documents/Repos/LLMDeepLearning/myvenv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 789, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/Users/LearningUser/Documents/Repos/LLMDeepLearning/myvenv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 536, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/LearningUser/Documents/Repos/LLMDeepLearning/myvenv/lib/python3.12/site-packages/urllib3/connection.py", line 507, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.12/3.12.7/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/opt/hom

In [181]:
from IPython.display import Markdown
Markdown(result)

# The Importance of Mental Health: Breaking the Stigma

## Introduction

In an era where physical health is often prioritized, mental health remains a topic that is frequently sidelined. However, mental health is as crucial as physical health, and it is time we address it with the seriousness it deserves. The importance of mental health cannot be overstated; it influences how we think, feel, and act in our daily lives. Addressing mental health issues can lead to a more fulfilling and balanced life. [Source](https://www.mentalhealth.gov/basics/what-is-mental-health).

## Understanding Mental Health

Mental health encompasses our emotional, psychological, and social well-being. It affects how we handle stress, relate to others, and make choices. Good mental health is more than just the absence of mental health problems; it involves maintaining a positive state of mind, managing life’s challenges effectively, and contributing to the community [Source](https://www.who.int/news-room/fact-sheets/detail/mental-health-strengthening-our-response).

### The Prevalence of Mental Health Issues

Mental health issues are more common than many realize. According to the World Health Organization, one in four people in the world will be affected by mental or neurological disorders at some point in their lives. This statistic underscores the need for increased awareness and better mental health services [Source](https://www.who.int/news-room/fact-sheets/detail/mental-health-strengthening-our-response).

## Breaking the Stigma

Despite its prevalence, mental health remains shrouded in stigma. Many people hesitate to seek help due to fear of judgment or discrimination. This stigma can be more damaging than the mental health issue itself, leading to a cycle of silence and suffering [Source](https://www.mentalhealth.org.uk/publications/stigma-and-discrimination-mental-health).

### The Role of Society

Society plays a significant role in either perpetuating or breaking the stigma surrounding mental health. Open conversations, education, and media representation can help normalize mental health issues and encourage individuals to seek help without fear of judgment [Source](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1489832/).

## Seeking Help: It's Okay Not to Be Okay

One of the most important aspects of mental health is recognizing when to seek help. Professional help, such as therapy or counseling, can provide individuals with the tools they need to manage their mental health effectively. Self-care practices, such as exercise, meditation, and maintaining a healthy lifestyle, can also play a crucial role in mental well-being [Source](https://www.psychologytoday.com/us/basics/counseling).

### Resources and Support

There are numerous resources available for those struggling with mental health issues. Many organizations provide free or low-cost services, including hotlines, online therapy platforms, and community support groups. Knowledge of these resources can empower individuals to take the first step toward mental well-being [Source](https://www.samhsa.gov/find-help/national-helpline).

## Conclusion

Mental health is a vital component of overall well-being, and addressing it should be a priority for everyone. By understanding mental health, breaking the stigma, and encouraging help-seeking behaviors, we can create a society where mental health is recognized and valued. Remember, it's okay not to be okay, and seeking help is a sign of strength, not weakness.

## References

- [What Is Mental Health?](https://www.mentalhealth.gov/basics/what-is-mental-health)
- [Mental Health: Strengthening Our Response](https://www.who.int/news-room/fact-sheets/detail/mental-health-strengthening-our-response)
- [Stigma and Discrimination of Mental Health](https://www.mentalhealth.org.uk/publications/stigma-and-discrimination-mental-health)
- [The Role of Society in Mental Health](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1489832/)
- [Counseling Basics](https://www.psychologytoday.com/us/basics/counseling)
- [National Helpline](https://www.samhsa.gov/find-help/national-helpline)

In [182]:
print(result)

# The Importance of Mental Health: Breaking the Stigma

## Introduction

In an era where physical health is often prioritized, mental health remains a topic that is frequently sidelined. However, mental health is as crucial as physical health, and it is time we address it with the seriousness it deserves. The importance of mental health cannot be overstated; it influences how we think, feel, and act in our daily lives. Addressing mental health issues can lead to a more fulfilling and balanced life. [Source](https://www.mentalhealth.gov/basics/what-is-mental-health).

## Understanding Mental Health

Mental health encompasses our emotional, psychological, and social well-being. It affects how we handle stress, relate to others, and make choices. Good mental health is more than just the absence of mental health problems; it involves maintaining a positive state of mind, managing life’s challenges effectively, and contributing to the community [Source](https://www.who.int/news-room/fact-sh